# Describing data, crosstabs, and running a regression: practice notebook

**Causal and Predictive Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

All the Python from the module's web page, plus short exercises (cells marked `# YOUR CODE HERE`). Four small data sets:

| File | What it is |
|---|---|
| `diamonds.csv` | 380 diamonds: `color` (1 = colorless or near-colorless), `carats`, `price` |
| `titanic.csv` | the 2,201 people on the Titanic: `class`, `age`, `gender`, `survival` |
| `cps99.csv` | 6,050 full-time workers (Current Population Survey, 1999): `Hours`, `Educ`, `Earnings`, `Race`, `Gender` |
| `wines.csv` | 230 US Chardonnays: rating `Score`, `Price`, `Year` |

**In Colab:** *File → Save a copy in Drive* first, so your changes are kept. Run cells with **Shift+Enter**.

## 0. Setup and data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

pd.options.display.float_format = "{:,.3f}".format

In [ ]:
from pathlib import Path

SITE = "https://tianlixia.com/causal-analytics/GBA472/A2/"   # the module's web address; filled in when the notebook is published, so the data load in Colab


def load(name):
    """Read a data file from the folder next to this notebook, or from the module's website."""
    for folder in [Path("data"), Path("web/data"), Path("../web/data")]:
        if (folder / name).exists():
            return pd.read_csv(folder / name)
    return pd.read_csv(SITE + "data/" + name)


diamonds = load("diamonds.csv")
titanic = load("titanic.csv")
cps = load("cps99.csv")
wines = load("wines.csv")
print(diamonds.shape, titanic.shape, cps.shape, wines.shape)
diamonds.head()

## Part A. Describing data

`describe()` gives the count, mean, standard deviation, minimum, quartiles and maximum of every numeric column. A histogram shows the shape; a scatter plot shows how two variables move together; `corr()` puts a number on it.

In [ ]:
diamonds.describe()

In [ ]:
diamonds["price"].median(), diamonds["price"].std()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
diamonds["price"].hist(bins=13, ax=ax[0], color="#2563eb", edgecolor="white")
ax[0].set_xlabel("price ($)"); ax[0].set_ylabel("number of diamonds")
diamonds["carats"].hist(bins=13, ax=ax[1], color="#2563eb", edgecolor="white")
ax[1].set_xlabel("carats")
plt.show()

In [ ]:
diamonds.plot.scatter(x="carats", y="price", s=12, alpha=0.6)
plt.show()
diamonds[["carats", "price"]].corr()

Categorical variables need different tools: `value_counts()` for one, a crosstab for two (Part B).

In [ ]:
titanic["class"].value_counts()

In [ ]:
titanic["class"].value_counts(normalize=True).round(3)

**Exercise A.** Describe the wine data: the summary statistics of `Score` and `Price`, a histogram of `Price`, a scatter plot of `Price` against `Score`, and their correlation. Is the association stronger or weaker than for the diamonds?

In [ ]:
# YOUR CODE HERE

## Part B. Crosstabs: conditional means for categorical data

A crosstab (Excel's PivotTable) counts the people in every combination of two categorical variables, or averages an outcome within each cell. With a 0/1 outcome the average is a share: the **survival rate** is a conditional mean, $E(\text{survived} \mid \text{class}, \text{gender})$.

In [ ]:
titanic["survived"] = (titanic["survival"] == "yes").astype(int)
pd.crosstab(titanic["class"], titanic["gender"])          # how many people in each cell

In [ ]:
titanic.pivot_table(index="class", columns="gender", values="survived", aggfunc="mean").round(3)   # survival rates

In [ ]:
# the same with the row and column totals ("Grand Total" in Excel), as percentages
(100 * titanic.pivot_table(index="class", columns="gender", values="survived", aggfunc="mean", margins=True)).round(0)

In [ ]:
# one conditioning variable at a time
print(titanic.groupby("gender")["survived"].mean().round(3))
print(titanic.groupby("age")["survived"].mean().round(3))

The CPS data have a numeric outcome (annual earnings). The same `pivot_table` call gives the **mean earnings** in every education-by-gender cell.

In [ ]:
cps["educ_group"] = pd.cut(cps["Educ"], [0, 11.9, 12, 14, 16, 18],
                           labels=["Less than high school", "High school", "Some college", "Bachelor's", "Graduate degree"])
earn = cps.pivot_table(index="educ_group", columns="Gender", values="Earnings", aggfunc="mean", observed=True).round(0)
earn["Women/Men"] = (earn["Female"] / earn["Male"]).round(2)
earn

In [ ]:
# cell sizes: how many workers stand behind each mean?
cps.pivot_table(index="educ_group", columns="Gender", values="Earnings", aggfunc="size", observed=True)

In [ ]:
# every value of Educ as its own row: small cells, and one outlier (9 years of education, men)
cps.pivot_table(index="Educ", columns="Gender", values="Earnings", aggfunc=["mean", "size", "max"]).round(0).head(8)

**Exercise B1.** Survival by age and gender: were children (`age == "young"`) more likely to survive than adults, for boys and for girls separately? Show the rates and the number of people in each cell.

In [ ]:
# YOUR CODE HERE

**Exercise B2.** Mean weekly hours by education group and gender (`Hours`). Do men work more hours in every education group? Could hours explain the earnings gap?

In [ ]:
# YOUR CODE HERE

## Part C. Running a regression: four ways in Python

Excel has four ways to get the least-squares line (Solver, the formulas, `INTERCEPT`/`SLOPE`, the Regression tool). Python has the same four, and the last one is the one to use.

In [ ]:
x, y = diamonds["carats"], diamonds["price"]

# 1. The formulas: slope = cov(x, y) / var(x); intercept = mean(y) - slope * mean(x)
slope = x.cov(y) / x.var()
intercept = y.mean() - slope * x.mean()
print(f"formulas:   intercept {intercept:,.2f}, slope {slope:,.2f}")

# 2. The sum of squared residuals, minimized numerically (what Solver does)
from scipy.optimize import minimize
ssr = lambda b: ((y - b[0] - b[1] * x) ** 2).sum()
res = minimize(ssr, x0=[1000, 1000], method="Nelder-Mead", options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 5000})
print(f"minimize:   intercept {res.x[0]:,.2f}, slope {res.x[1]:,.2f}, SSR {res.fun:,.0f}")
print(f"SSR at the starting guess (1000, 1000): {ssr([1000, 1000]):,.0f}")

# 3. numpy's one-liner (like INTERCEPT and SLOPE)
b1, b0 = np.polyfit(x, y, 1)
print(f"polyfit:    intercept {b0:,.2f}, slope {b1:,.2f}")

In [ ]:
# 4. statsmodels: the full regression table (like Excel's Regression tool)
model = smf.ols("price ~ carats", data=diamonds).fit()
print(model.summary())

Reading the table: the `coef` column holds the two estimates (Excel's *Coefficients*); `std err`, `t`, `P>|t|` and the interval are the inference columns, and `R-squared` the fit. The rest of the course explains them. For now, the line is

$$\widehat{\text{price}} = -1{,}679 + 5{,}573 \times \text{carats}.$$

In [ ]:
print(model.params.round(2))
print("predicted price at 1.0 and 1.2 carats:", model.predict(pd.DataFrame({"carats": [1.0, 1.2]})).round(0).tolist())
print("sum of squared residuals:", f"{model.ssr:,.0f}", "  R-squared:", round(model.rsquared, 3))

In [ ]:
ax = diamonds.plot.scatter(x="carats", y="price", s=12, alpha=0.5)
grid = np.linspace(0.7, 1.3, 2)
ax.plot(grid, model.params["Intercept"] + model.params["carats"] * grid, color="#ea580c", lw=2.5)
plt.show()

**Exercise C1.** Run the wine regression, `Price ~ Score`. Interpret the slope in a sentence, and predict the price of a 90-point wine. Check the slope with the formula `cov / var`.

In [ ]:
# YOUR CODE HERE

**Exercise C2.** A regression on a dummy. Regress `survived` on a 0/1 variable `female` in the Titanic data. Compare the intercept and the slope with the two survival rates from Part B. What does the slope equal?

In [ ]:
titanic["female"] = (titanic["gender"] == "female").astype(int)
# YOUR CODE HERE

---
*That's all. The live sessions take the regression table apart: what the standard errors, t-statistics and R-squared mean, and what happens with more than one X.*